In [1]:
pip install cooper-optim

  Using cached cooper_optim-1.0.1-py3-none-any.whl.metadata (12 kB)
  Using cached torch-2.11.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.3.0-py3-none-any.whl.metadata (10 kB)
  Using cached cuda_toolkit-13.0.2-py2.py3-none-any.whl.metadata (9.4 kB)
  Using cached cuda_bindings-13.2.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (2.3 kB)
  Using cached nvidia_cudnn_cu13-9.19.0.56-py3-none-manylinux_2_27_x86_64.whl.metadata (1.9 kB)
  Using cached nvidia_cusparselt_cu13-0.8.0-py3-none-manylinux2014_x86_64.whl.metadata (12 kB)
  Using cached nvidia_nccl_cu13-2.28.9-py3-none-manylinux_2_18_x86_64.whl.metadata (2.0 kB)
  Using cached nvidia_nvshmem_cu13-3.4.5-py3-non

In [4]:
import numpy as np
import torch
import torch.nn as nn
import cooper

In [5]:
def compute_d(arch):
    d = 0
    for i in range(len(arch) - 1):
        d += arch[i] * arch[i + 1]
    return d

In [6]:
def unpack_weights(w_flat, arch):
    matrices = []
    idx = 0
    for i in range(len(arch) - 1):
        n_in, n_out = arch[i], arch[i + 1]
        size = n_in * n_out
        W = w_flat[idx: idx + size].reshape(n_out, n_in)
        matrices.append(W)
        idx += size
    return matrices

In [8]:
def forward_pass(u, w_flat, biases, arch):
    weight_mats = unpack_weights(w_flat, arch)
    a = u
    for l, W in enumerate(weight_mats):
        z = W @ a + biases[l].unsqueeze(1)
        if l < len(weight_mats) - 1:
            a = torch.tanh(z)
        else:
            a = z
    return a

In [10]:
def binary_entropy(Q, eps=1e-6):
    Qc = Q.clamp(eps, 1 - eps) # 
    return -(Qc * Qc.log() + (1 - Qc) * (1 - Qc).log()).sum()

In [ ]:
class SparseNNModel(nn.Module):
    def __init__(self, arch, k, eps=1e-6):
        super().__init__()
        self.arch = arch
        self.k = k
        self.eps = eps
        self.d = compute_d(arch)

        # Q initialized column-stochastic
        Q_init = torch.ones(self.d, k) / self.d
        self.Q = nn.Parameter(Q_init)

        # reduced coefficients
        self.x_coeff = nn.Parameter(torch.randn(k) * 0.1)

        # biases
        self.biases = nn.ParameterList()
        for i in range(len(arch) - 1):
            self.biases.append(nn.Parameter(torch.zeros(arch[i + 1])))

    def get_w(self):
        return self.Q @ self.x_coeff

    def forward(self, u):
        return forward_pass(u, self.get_w(), self.biases, self.arch)

    def clamp_Q(self):
        with torch.no_grad():
            self.Q.clamp_(self.eps, 1.0 - self.eps) # 

In [13]:
Q = torch.ones(120,20) / 120
x = torch.randn(20) * 0.1
w = Q @ x
print(w)

tensor([0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051,
        0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051,
        0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051,
        0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051,
        0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051,
        0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051,
        0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051,
        0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051,
        0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051,
        0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051,
        0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051,
        0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051, 0.0051,
        0.0051, 0.0051, 0.0051, 0.0051, 

In [ ]:
class SparseNNCMP(cooper.ConstrainedMinimizationProblem):
    def __init__(
        self,
        model,
        c0,
        u_data,
        y_data,
        init_penalty_entropy=1.0, # hyperparameter for entropy penalty
        init_penalty_colsum=5.0, # hyperparameter for column sum penalty
        tol=1e-6, 
    ):
        super().__init__()
        self.model = model
        self.c0 = c0
        self.u_data = u_data
        self.y_data = y_data
        self.penalty_entropy = init_penalty_entropy
        self.penalty_colsum = init_penalty_colsum

        # H(Q) - c0 <= 0
        self.entropy_eq = cooper.Constraint(
            constraint_type=cooper.ConstraintType.INEQUALITY,
            formulation_type=cooper.formulations.AugmentedLagrangian,
            multiplier=cooper.multipliers.DenseMultiplier(num_constraints=1),
            penalty_coefficient=cooper.penalty_coefficients.DensePenaltyCoefficient(
                init=torch.tensor([init_penalty_entropy], dtype=torch.float32)
            ),
        )

        # Q^T 1_d - 1_k - tol <= 0
        self.colsum_eq_upper = cooper.Constraint(
            constraint_type=cooper.ConstraintType.INEQUALITY,
            formulation_type=cooper.formulations.AugmentedLagrangian,
            multiplier=cooper.multipliers.DenseMultiplier(num_constraints=model.k),
            penalty_coefficient=cooper.penalty_coefficients.DensePenaltyCoefficient(
                init=torch.full((model.k,), float(init_penalty_colsum), dtype=torch.float32)
            ),
        )

        # -Q^T 1_d + 1_k - tol <= 0
        self.colsum_eq_lower = cooper.Constraint(
            constraint_type=cooper.ConstraintType.INEQUALITY,
            formulation_type=cooper.formulations.AugmentedLagrangian,
            multiplier=cooper.multipliers.DenseMultiplier(num_constraints=model.k),
            penalty_coefficient=cooper.penalty_coefficients.DensePenaltyCoefficient(
                init=torch.full((model.k,), float(init_penalty_colsum), dtype=torch.float32)
            ),
        )


    def compute_cmp_state(self):
        m = self.u_data.shape[1]
        y_pred = self.model(self.u_data)
        loss = ((self.y_data - y_pred) ** 2).sum() / m

        H = binary_entropy(self.model.Q, eps=self.model.eps)
        entropy_violation = (H - self.c0).reshape(1) # (1,)

        colsum_violation_upper = self.model.Q.sum(dim=0) - 1.0 # (20,)
        colsum_violation_lower = 1.0 - self.model.Q.sum(dim=0) # (20,)
        
        return cooper.CMPState(
            loss=loss,
            observed_constraints={
                self.entropy_eq: cooper.ConstraintState(violation=entropy_violation),
                self.colsum_eq_upper: cooper.ConstraintState(violation=colsum_violation_upper),
                self.colsum_eq_lower: cooper.ConstraintState(violation=colsum_violation_lower),
            },
        )
    
    # Debug function to print column gradient components
    def print_column_gradients(self):
